# 5 — Stage 2: which words are the drug and which are the effect?

Stage 1 answered *does this sentence report an adverse drug event?* Stage 2 answers the
follow-up: **which exact words?**

> A case of toxic hepatitis caused by methotrexate.
>            └─── EFFECT ───┘         └── DRUG ──┘

This is **sequence labelling**: one decision per word, not one per sentence. The model is
the same BiLSTM encoder as Stage 1, with two changes — one output per word instead of one
per sentence, and a **CRF** on top.

> ### About running this notebook
>
> **The training code below is real and complete** — it is the code that produced
> `models/stage2/run10_crf/`. It sits behind a `TRAIN` switch that is **off** by default.
> With `TRAIN = False` the notebook loads the finished checkpoint and scores it on this
> laptop's CPU in under a second.

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", ROOT)

project root: E:\CSE\NLP Project


---

## 5.1 The labels

Stage 2 trains only on sentences that really are ADEs — 2,990 of them — because those are
the only ones the corpus annotates with spans. Each sentence's character spans become one
BIO tag per word (notebook 2 covers the conversion; here is what comes out).

In [2]:
from src.bio import to_bio, TAGS, TAG_TO_ID, count_illegal_transitions
from src.vocab import is_indexable

test = pd.read_parquet(ROOT / "data" / "splits" / "stage2_test.parquet")
print(f"{len(test)} test sentences, {int(test['n_spans'].sum()):,} annotated spans")
print(f"tag inventory: {TAGS}\n")

row = test.iloc[0]
tokens, tags = to_bio(row["text"], [tuple(s) for s in json.loads(row["spans"])])
print(row["text"], "\n")
print("  " + "  ".join(f"{t:>14s}" for t in tokens))
print("  " + "  ".join(f"{t:>14s}" for t in tags))

640 test sentences, 1,636 annotated spans
tag inventory: ('O', 'B-DRUG', 'I-DRUG', 'B-EFFECT', 'I-EFFECT')

2-CdA induces lymphocytopenia, which may explain the improvement in this patient's psoriasis. 

           2-cda         induces  lymphocytopenia               ,           which             may         explain             the     improvement              in            this       patient's       psoriasis               .
          B-DRUG               O        B-EFFECT               O               O               O               O               O               O               O               O               O               O               O


That is the training signal. The model sees the words and has to produce the tags.

### Preparing a split

Punctuation is dropped — it is not in the shared vocabulary — and the tags have to be
dropped **in lockstep** with it, or every sentence containing a comma is shifted by one
from that comma onward. It does not raise; it just trains on wrong labels. That is why
`encode_tokens_with_tags` does both at once instead of leaving it to the caller.

In [3]:
from src.bio import ConversionStats

def load_word_level(split):
    """Parquet -> [{words, tags, text}], at WORD level.

    The conversion is redone from the committed spans every time rather than
    cached, so the converter and the training data can never drift apart - it
    costs about a second for 4,271 sentences.
    """
    df = pd.read_parquet(ROOT / "data" / "splits" / f"stage2_{split}.parquet")
    stats = ConversionStats()
    rows = []

    for text, spans in zip(df["text"], df["spans"]):
        raw = json.loads(spans) if isinstance(spans, str) else spans
        parsed = [(int(s), int(e), str(label)) for s, e, label in raw]

        tokens, tags = to_bio(text, parsed, stats=stats, strict=False)
        kept = [(t, tag) for t, tag in zip(tokens, tags) if is_indexable(t)]
        if not kept:
            continue

        words, word_tags = zip(*kept)
        rows.append({"words": list(words), "tags": list(word_tags), "text": text})

    return rows, stats


rows, stats = load_word_level("test")
words = [r["words"] for r in rows]
gold = [r["tags"] for r in rows]

print(f"{len(rows)} sentences, {sum(len(w) for w in words):,} words")
print(f"gold entities: {sum(1 for tags in gold for t in tags if t.startswith('B-')):,}")

# This one matters for section 5.2: the gold data contains no structurally
# impossible tag sequence at all.
print(f"illegal transitions in the GOLD tags: "
      f"{sum(count_illegal_transitions(g) for g in gold)}")

640 sentences, 11,452 words
gold entities: 1,612
illegal transitions in the GOLD tags: 0


---

## 5.2 The model, and what the CRF is for

```
word ids -> Embedding (E3) -> BiLSTM -> linear -> 5 scores per word
                                                       |
                                              CRF: Viterbi over the whole sentence
```

A plain per-word softmax decides each position **independently**. Nothing stops it
emitting `I-DRUG` straight after `O` — a sequence that cannot describe any entity at all.
The gold data contains exactly zero of those (checked just above), so they are always
errors.

A **CRF** learns a score for every tag-to-tag transition and decodes the whole sentence
jointly with Viterbi, so a path through a transition that never appears in the gold data
becomes very unlikely.

Be precise about the mechanism, because it is easy to overclaim:

> `pytorch-crf` **forbids nothing**. Its transition matrix starts uniform and every
> transition stays reachable. Because the gold sequences contain no illegal transition,
> training drives those transition scores down until Viterbi rarely picks a path through
> them. The constraint is **learned from the data, not imposed by the architecture** — so
> a few still get through when the word-level evidence is strong enough.

In [4]:
import numpy as np
from src.models import BiLSTMTagger

MATRICES = ROOT / "models" / "emb_matrices"
matrix = np.load(MATRICES / "E3.npy")

plain = BiLSTMTagger(matrix, use_crf=False)
crf = BiLSTMTagger(matrix, use_crf=True)

print(f"without CRF: {plain.trainable_parameters():,} trainable parameters")
print(f"with CRF   : {crf.trainable_parameters():,} trainable parameters")
print(f"the CRF is : {crf.trainable_parameters() - plain.trainable_parameters()} of them "
      f"- a {len(TAGS)}x{len(TAGS)} transition matrix, plus start and end scores")

without CRF: 1,145,349 trainable parameters
with CRF   : 1,145,384 trainable parameters
the CRF is : 35 of them - a 5x5 transition matrix, plus start and end scores


**35 extra parameters.** That is the entire CRF.

It is not free, though: the forward algorithm and Viterbi decoding are sequential over the
sentence, so training took about 5x as long as the plain version.

---

## 5.3 Training

The switch below controls the training cell.

In [5]:
# ---------------------------------------------------------------------------
# OFF by default. The checkpoint is already in models/stage2/run10_crf/;
# section 5.4 loads it. Set True in a Kaggle GPU session to retrain.
# ---------------------------------------------------------------------------
TRAIN = False

from src.utils import DEFAULT_SEED, get_device_count, log_run, pin_single_gpu, set_seed

OUT = ROOT / "models" / "stage2"
MAX_LEN = 96          # covers 100% of Stage 2 sentences; p95 is 38 tokens
PATIENCE = 4

print(f"TRAIN = {TRAIN} | visible GPUs: {get_device_count()}")

TRAIN = False | visible GPUs: 0


In [6]:
def train_tagger(run_id="10", *, use_crf=True, embedding="E3", epochs=30, batch_size=32,
                 lr=1e-3, hidden_dim=256, dropout=0.5, freeze_embeddings=True,
                 seed=DEFAULT_SEED, dataset_version=""):
    """Train the Stage 2 tagger. Run 10 is this with `use_crf=True`."""
    import random
    import time

    import numpy as np
    import torch

    from src.bio import entity_metrics, token_accuracy
    from src.models import BiLSTMTagger
    from src.vocab import encode_tokens_with_tags, load_vocab

    set_seed(seed)
    device_count = get_device_count()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if device_count > 1:
        # DataParallel changes both the effective batch AND how the CRF loss is
        # reduced, so it is refused rather than silently handled.
        raise SystemExit(
            f"{device_count} GPUs visible. Call pin_single_gpu() before importing torch.")

    vocab, index = load_vocab(MATRICES / "vocab.json")
    matrix = np.load(MATRICES / f"{embedding}.npy")
    if matrix.shape[0] != len(vocab):
        raise ValueError(f"{embedding}.npy has {matrix.shape[0]} rows but vocab.json "
                         f"has {len(vocab)} - different builds.")

    splits = {}
    for split in ("train", "dev", "test"):
        split_rows, split_stats = load_word_level(split)
        encoded = []
        for r in split_rows:
            ids, tags = encode_tokens_with_tags(r["words"], r["tags"], index)
            if ids:
                encoded.append((ids[:MAX_LEN], tags[:MAX_LEN], r["text"]))
        splits[split] = encoded
        print(f"  stage2_{split}: {len(encoded):,} sentences | {split_stats.spans:,} spans "
              f"| {split_stats.entities_tagged:,} tagged")

    def collate(chunk):
        width = max(len(ids) for ids, _, _ in chunk)
        id_matrix, tag_matrix, lengths = [], [], []
        for ids, tags, _ in chunk:
            pad = width - len(ids)
            id_matrix.append(ids + [0] * pad)
            # Padded tag slots are never scored: the softmax branch masks them
            # to IGNORE_INDEX and the CRF excludes them via the mask.
            tag_matrix.append([TAG_TO_ID[t] for t in tags] + [0] * pad)
            lengths.append(len(ids))
        return (torch.tensor(id_matrix, dtype=torch.long),
                torch.tensor(lengths, dtype=torch.long),
                torch.tensor(tag_matrix, dtype=torch.long))

    def batches(data, shuffle=False, epoch_seed=seed):
        order = list(range(len(data)))
        if shuffle:
            random.Random(epoch_seed).shuffle(order)
        for start in range(0, len(order), batch_size):
            chunk = [data[i] for i in order[start:start + batch_size]]
            yield collate(chunk), chunk

    def predict(data):
        """Decode every sentence -> (gold, predicted) ragged tag strings."""
        model.eval()
        gold_all, pred_all = [], []
        for (ids, lengths, _), chunk in batches(data):
            decoded = model.decode(ids.to(device), lengths.to(device))
            for (_, gold_tags, _), path in zip(chunk, decoded):
                pred_all.append([TAGS[i] for i in path])
                gold_all.append(list(gold_tags))
        return gold_all, pred_all

    print(f"\nrun {run_id} | {embedding} | {'BiLSTM-CRF' if use_crf else 'BiLSTM softmax'}")
    print(f"  device {device} | batch {batch_size} | tags {list(TAGS)}")

    model = BiLSTMTagger(matrix, num_tags=len(TAGS), hidden_dim=hidden_dim,
                         dropout=dropout, use_crf=use_crf,
                         freeze_embeddings=freeze_embeddings).to(device)
    print(f"  trainable params {model.trainable_parameters():,}")

    optimiser = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    best = {"dev_f1": -1.0, "epoch": 0, "state": None}
    history, t0 = [], time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running = count = 0
        for (ids, lengths, tags), _ in batches(splits["train"], shuffle=True,
                                               epoch_seed=seed + epoch):
            optimiser.zero_grad()
            # With a CRF the model computes its own loss inside forward(),
            # because the probability of a tag SEQUENCE is not the product of
            # independent per-word probabilities.
            loss = model(ids.to(device), lengths.to(device), tags.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimiser.step()
            running += loss.item()
            count += 1

        dev = entity_metrics(*predict(splits["dev"]))
        dev_f1 = dev["entity_f1_strict"]
        history.append({"epoch": epoch, "train_loss": running / count,
                        "dev_entity_f1_strict": dev_f1,
                        "dev_illegal": dev["illegal_transitions"]})

        marker = ""
        if dev_f1 > best["dev_f1"]:
            best = {"dev_f1": dev_f1, "epoch": epoch,
                    "state": {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}}
            marker = "  <- best"
        print(f"  epoch {epoch:>2}  loss {running / count:>8.4f}  "
              f"dev entity-F1 {dev_f1:.4f}  illegal {dev['illegal_transitions']:>3}{marker}")

        if epoch - best["epoch"] >= PATIENCE:
            print(f"  early stop: {PATIENCE} epochs without improvement")
            break

    train_secs = time.time() - t0

    model.load_state_dict(best["state"])
    gold_tags, pred_tags = predict(splits["test"])
    metrics = entity_metrics(gold_tags, pred_tags)
    metrics.update({"token_accuracy": token_accuracy(gold_tags, pred_tags),
                    "dev_entity_f1_strict": best["dev_f1"], "best_epoch": best["epoch"],
                    "epochs_run": len(history), "train_seconds": round(train_secs, 1)})

    print(f"\n  TEST entity-F1 strict {metrics['entity_f1_strict']:.4f} "
          f"| lenient {metrics['entity_f1_lenient']:.4f}")
    print(f"  illegal transitions {metrics['illegal_transitions']} in "
          f"{metrics['illegal_sentences']} sentences | {train_secs:.0f}s")

    out = OUT / f"run{run_id}_{'crf' if use_crf else 'softmax'}"
    out.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": best["state"], "config": model.config,
                "embedding": embedding, "run_id": run_id, "tags": list(TAGS)},
               out / "checkpoint.pt")
    (out / "test_predictions.json").write_text(
        json.dumps({"gold": gold_tags, "pred": pred_tags,
                    "text": [t for _, _, t in splits["test"]]}), encoding="utf-8")
    (out / "metrics.json").write_text(
        json.dumps({"metrics": metrics, "history": history}, indent=2), encoding="utf-8")

    log_run(run_id=run_id, stage="2",
            model="bilstm_crf" if use_crf else "bilstm_softmax",
            embedding=embedding, metrics=metrics,
            params={**model.config, "max_len": MAX_LEN, "patience": PATIENCE,
                    "optimiser": "adam", "grad_clip": 5.0,
                    "early_stop_on": "dev entity_f1_strict", "tags": list(TAGS),
                    "torch": torch.__version__},
            seed=seed, dataset_version=dataset_version,
            per_device_batch=batch_size, device_count=device_count,
            epochs=len(history), lr=lr,
            notes=f"Run {run_id}; BiLSTM-CRF on {embedding}; best epoch {best['epoch']}.")
    print(f"  wrote {out}")


if TRAIN:
    pin_single_gpu()                       # before torch is imported
    train_tagger("10", use_crf=True)

---

## 5.4 The result

This is what runs with `TRAIN = False`. The checkpoint is loaded from `models/stage2/`, run
over the test split on this laptop's CPU, and scored.

**Entity-F1, not word accuracy.** Getting 9 words out of 10 right sounds good until the one
you missed was half the drug name — an entity with the wrong boundary is simply wrong, not
90% right. So scoring is *strict*: a predicted entity counts only if its label **and** both
its boundaries match a gold entity exactly.

(`O` is about 79% of all words, so word accuracy has the same problem accuracy had in
Stage 1 — it is reported and then argued against.)

In [7]:
import time
from src.bio import entity_metrics, token_accuracy
from src.pipeline import load_tagger

t0 = time.perf_counter()
predicted = load_tagger().tag(words)
elapsed = time.perf_counter() - t0

scores = entity_metrics(gold, predicted)
scores["token_accuracy"] = token_accuracy(gold, predicted)

print(f"{len(words)} sentences tagged in {elapsed:.1f}s on CPU\n")
for key in ["entity_f1_strict", "entity_precision_strict", "entity_recall_strict",
            "entity_f1_lenient", "drug_f1", "effect_f1", "token_accuracy"]:
    print(f"  {key:26s} {scores[key]:.4f}")
print(f"\n  {'gold entities':26s} {scores['gold_entities']:,}")
print(f"  {'predicted entities':26s} {scores['pred_entities']:,}")
print(f"  {'illegal transitions':26s} {scores['illegal_transitions']} "
      f"(in {scores['illegal_sentences']} sentences)")

640 sentences tagged in 0.5s on CPU

  entity_f1_strict           0.8308
  entity_precision_strict    0.8540
  entity_recall_strict       0.8089
  entity_f1_lenient          0.8314
  drug_f1                    0.9040
  effect_f1                  0.7629
  token_accuracy             0.9417

  gold entities              1,612
  predicted entities         1,532
  illegal transitions        5 (in 5 sentences)


### Reading these numbers

**Strict entity-F1 0.8308** is the headline: roughly five entities in six are found with
exactly the right label and exactly the right boundaries.

**DRUG is much easier than EFFECT** — 0.904 against 0.763. That gap is the interesting
part. Drug names are mostly single tokens drawn from a finite vocabulary the embeddings
know well. Effects are phrases (`acute renal failure`, `labial angioedema`) whose
boundaries are genuinely ambiguous, and strict scoring gives no credit for getting four
words of a five-word span right.

**5 illegal transitions** out of 1,532 predicted entities. That is the CRF doing its job —
and note it is 5, not 0, exactly as "learned from the data, not imposed" predicts.

**Token accuracy is 0.94 and means very little.** `O` is most of the data. It is here to be
argued against, the same way accuracy was in Stage 1.

---

## 5.5 Seeing it tag

Strict scoring is unforgiving, so it is worth looking at what the output actually is.

In [8]:
from src.bio import strict_entities
from src.pipeline import words_of

tagger = load_tagger()
sentences = [
    "A case of toxic hepatitis caused by methotrexate and etretinate is presented.",
    "Severe rhabdomyolysis developed after starting simvastatin.",
    # A real test sentence, and then the same content compressed into a shorter one.
    "A 55-year-old woman presented an episode of acute urticaria and labial angioedema "
    "60 minutes after ingesting 500 mg of cloxacillin for a skin abscess.",
    "Acute urticaria and labial angioedema followed ingestion of cloxacillin.",
]

for sentence in sentences:
    tokens = words_of(sentence)
    tags = tagger.tag([tokens])[0]
    print(sentence)
    for a, b, label in strict_entities(tags):
        print(f"    {label:7s} {' '.join(tokens[a:b])}")
    print()

A case of toxic hepatitis caused by methotrexate and etretinate is presented.
    EFFECT  toxic hepatitis
    DRUG    methotrexate
    DRUG    etretinate

Severe rhabdomyolysis developed after starting simvastatin.
    EFFECT  severe rhabdomyolysis
    DRUG    simvastatin

A 55-year-old woman presented an episode of acute urticaria and labial angioedema 60 minutes after ingesting 500 mg of cloxacillin for a skin abscess.
    EFFECT  acute urticaria
    EFFECT  labial angioedema
    DRUG    cloxacillin

Acute urticaria and labial angioedema followed ingestion of cloxacillin.
    EFFECT  acute urticaria
    DRUG    labial
    EFFECT  angioedema
    DRUG    cloxacillin



The first two show the model is not pattern-matching on word order: the effect comes after
the drug in one and before it in the other, and both are read correctly.

**The last two are the same content, and the model gets one of them wrong.** In the real
test sentence it reads `labial angioedema` as a single EFFECT. In the compressed
paraphrase it splits that phrase, tagging `labial` as a DRUG — which is not even the right
*kind* of thing. Nothing about the phrase changed; only the sentence around it did.

That is worth seeing before the scores in the next notebook. Strict entity-F1 of 0.83
means roughly one entity in six is wrong like this, and the failures are not evenly spread
— notebook 6 measures where they cluster.

---

## What this notebook produced

| Artefact | Contents |
|---|---|
| `models/stage2/run10_crf/` | the trained tagger |
| `results/runs.csv` | its row, written by `log_run` |

**Key numbers:** strict entity-F1 0.8308; DRUG 0.904 against EFFECT 0.763; 5 illegal
transitions.

Everything so far has scored Stage 2 on **gold** ADE sentences, as though Stage 1 were
perfect. **Next:** [6 — The pipeline](06_pipeline_and_demo.ipynb), where the two stages are
chained and the cost of that assumption is measured.